# Importation et nettoyage

In [5]:
import sys
!{sys.executable} -m pip install patsy

Defaulting to user installation because normal site-packages is not writeable
  Using cached patsy-1.0.2-py2.py3-none-any.whl.metadata (3.6 kB)
Using cached patsy-1.0.2-py2.py3-none-any.whl (233 kB)


In [ ]:
import pandas as pd
from patsy import dmatrix
don = pd.read_table("DONNEES/ozone.txt", header=0, sep=";")
don.head(3)

,Date,O3,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebu,vent
0,19960422,63.6,13.4,15.0,7,0,0,3,0,9.35,95.6,NUAGE,EST
1,19960429,89.6,15.0,15.7,4,3,0,0,0,5.40,100.2,SOLEIL,NORD
2,19960506,79.0,7.9,10.1,8,0,0,7,0,19.30,105.6,NUAGE,EST


La colonne Date n'est pas une variable donc on l'enlève du dataframe

In [7]:
don = don.drop(columns=["Date"])
don.head(3)

,O3,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebu,vent
0,63.6,13.4,15.0,7,0,0,3,0,9.35,95.6,NUAGE,EST
1,89.6,15.0,15.7,4,3,0,0,0,5.40,100.2,SOLEIL,NORD
2,79.0,7.9,10.1,8,0,0,7,0,19.30,105.6,NUAGE,EST


Je renomme la variable d'intérêt Y, je mets en général les X d'un côté et je regarde les types de variables afin de recoder les variables qualitatives

In [8]:
don = don.rename(columns={"O3":"Y"})
X = don.drop(columns=["Y"])
Y = don.filter(["Y"])

In [9]:
Xquanti = X.select_dtypes(exclude=['object'])
Xquali = X.select_dtypes(include=['object'])
print(Xquanti.head(3))
print(Xquali.head(3))

    T12   T15  Ne12  N12  S12  E12  W12     Vx    O3v
0  13.4  15.0     7    0    0    3    0   9.35   95.6
1  15.0  15.7     4    3    0    0    0   5.40  100.2
2   7.9  10.1     8    0    0    7    0  19.30  105.6
     nebu  vent
0   NUAGE   EST
1  SOLEIL  NORD
2   NUAGE   EST


Je vais recoder les variables qualitatives avec get_dummies de pd

In [10]:
XqualiD = pd.get_dummies(Xquali,drop_first=True,dtype=float)
XqualiD.head(3)

,nebu_SOLEIL,vent_NORD,vent_OUEST,vent_SUD
0,0.0,0.0,0.0,0.0
1,1.0,1.0,0.0,0.0
2,0.0,0.0,0.0,0.0


Je recombine

In [11]:
Xbase = pd.concat([Xquanti,XqualiD],axis=1)
dfbase = pd.concat([Xbase,Y],axis=1)
dfbase.to_csv("dfbase.csv",index=False)

dfbase.head()

,T12,T15,Ne12,N12,S12,E12,W12,Vx,O3v,nebu_SOLEIL,vent_NORD,vent_OUEST,vent_SUD,Y
0,13.4,15.0,7,0,0,3,0,9.35,95.6,0.0,0.0,0.0,0.0,63.6
1,15.0,15.7,4,3,0,0,0,5.40,100.2,1.0,1.0,0.0,0.0,89.6
2,7.9,10.1,8,0,0,7,0,19.30,105.6,0.0,0.0,0.0,0.0,79.0
3,13.1,11.7,7,7,0,0,0,12.60,95.2,0.0,1.0,0.0,0.0,81.2
4,14.1,16.0,6,0,0,0,6,-20.30,82.8,0.0,0.0,1.0,0.0,88.0


Si je veux faire des polynomes

In [12]:
X2 = Xquanti**2
X2 = X2.add_suffix("car")
X3 = Xquanti**3
X3 = X3.add_suffix("cub")
Xpoly = pd.concat([Xbase,X2,X3],axis=1)
dfpoly = pd.concat([Xpoly,Y],axis=1)
print(dfpoly.head(3))
dfpoly.to_csv("dfpoly.csv",index=False)

    T12   T15  Ne12  N12  S12  E12  W12     Vx    O3v  nebu_SOLEIL  ...  \
0  13.4  15.0     7    0    0    3    0   9.35   95.6          0.0  ...   
1  15.0  15.7     4    3    0    0    0   5.40  100.2          1.0  ...   
2   7.9  10.1     8    0    0    7    0  19.30  105.6          0.0  ...   

     T12cub    T15cub  Ne12cub  N12cub  S12cub  E12cub  W12cub        Vxcub  \
0  2406.104  3375.000      343       0       0      27       0   817.400375   
1  3375.000  3869.893       64      27       0       0       0   157.464000   
2   493.039  1030.301      512       0       0     343       0  7189.057000   

        O3vcub     Y  
0   873722.816  63.6  
1  1006012.008  89.6  
2  1177583.616  79.0  

[3 rows x 32 columns]


Si on veut faire des interactions plus compliqué, faire une boucle sur Xquanti sinon utiliser dmatrix de patsy

In [13]:
from patsy import dmatrix
nomvarexpl = list(don.columns.difference(["Y"]))
formule = "~" + "+".join(nomvarexpl)
dsX = dmatrix(formule,don,return_type="dataframe")
#je retire l'intercept
dsX = dsX.drop(columns=["Intercept"])
dfbase = pd.concat([dsX,Y],axis=1)
dfbase.to_csv("dfbase.csv",index=False)
dfbase.head(3)

,nebu[T.SOLEIL],vent[T.NORD],vent[T.OUEST],vent[T.SUD],E12,N12,Ne12,O3v,S12,T12,T15,Vx,W12,Y
0,0.0,0.0,0.0,0.0,3.0,0.0,7.0,95.6,0.0,13.4,15.0,9.35,0.0,63.6
1,1.0,1.0,0.0,0.0,0.0,3.0,4.0,100.2,0.0,15.0,15.7,5.40,0.0,89.6
2,0.0,0.0,0.0,0.0,7.0,0.0,8.0,105.6,0.0,7.9,10.1,19.30,0.0,79.0


Si on veut faire l'intéraction

In [14]:
formuleI = "1 + (" + "+".join(nomvarexpl)+")**2"
Xinter = dmatrix(formuleI,don,return_type="dataframe")
#je retire l'intercept
Xinter = Xinter.drop(columns=["Intercept"])
dfinter = pd.concat([Xinter,Y],axis=1)
dfinter.to_csv("dfinter.csv",index=False)
dfinter.head(3)

,nebu[T.SOLEIL],vent[T.NORD],vent[T.OUEST],vent[T.SUD],nebu[T.SOLEIL]:vent[T.NORD],nebu[T.SOLEIL]:vent[T.OUEST],nebu[T.SOLEIL]:vent[T.SUD],E12,E12:nebu[T.SOLEIL],E12:vent[T.NORD],...,S12:T15,S12:Vx,S12:W12,T12:T15,T12:Vx,T12:W12,T15:Vx,T15:W12,Vx:W12,Y
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,...,0.0,0.0,0.0,201.00,125.29,0.0,140.25,0.0,0.0,63.6
1,1.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,235.50,81.00,0.0,84.78,0.0,0.0,89.6
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,...,0.0,0.0,0.0,79.79,152.47,0.0,194.93,0.0,0.0,79.0
